# Aim 2 Phase 1 — cumulative feature ablation (Exp 1-4)

**Configs (cumulative):**
- exp1: physio + sample entropy
- exp2: exp1 + SpO₂ PSD
- exp3: exp2 + CPC-proxy
- exp4: exp3 + ECG band power (= full phase1batch-v1 schema)

Same FIXED LightGBM params as v8.5-tax / phase1_exp5/6 — measures pure feature-group contribution. Sleep-only filter, seed=42 outer split (matches all prior v6/v8/v8.5 work).

Runtime: high-RAM CPU, ~5–10 min per config, ~30–40 min total.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
assert os.path.exists(DRIVE_ROOT), f'Expected {DRIVE_ROOT} to exist'
print(f'Drive root OK: {DRIVE_ROOT}')

In [ ]:
# 2. Pull the repo from GitHub (private; PAT lives in Colab secret GITHUB_PAT_THESIS)
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT_THESIS')
REPO = 'jameswudo1019hack/thesis-bme-pipeline'
import subprocess, shutil, pathlib
if pathlib.Path('/content/Code').exists():
    subprocess.run(['git', '-C', '/content/Code', 'pull', '--ff-only'], check=True)
else:
    url = f'https://{PAT}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth=1', url, '/content/Code'], check=True)
print('Repo state:')
subprocess.run(['git', '-C', '/content/Code', 'log', '-1', '--oneline'], check=True)

In [ ]:
# 3. Pull features tar from Drive and extract to /content/Code/features
import shutil, subprocess, time
TAR = f'{DRIVE_ROOT}/data/features-phase1batch-v1.tar'
DST = '/content/Code/features'
assert os.path.exists(TAR), f'Tar not found at {TAR} — upload it first'
if os.path.exists(DST):
    shutil.rmtree(DST)
os.makedirs(DST, exist_ok=True)
t0 = time.time()
subprocess.run(['tar', '-xf', TAR, '-C', '/content/Code'], check=True)
print(f'Extract done in {time.time()-t0:.0f}s')
n_parquets = len([f for f in os.listdir(DST) if f.endswith('.parquet') and not f.startswith('._')])
print(f'  parquets: {n_parquets}')

In [ ]:
# 4. Install deps + thesis_pipeline (editable)
!pip install -q lightgbm pyarrow neurokit2 mne lxml click 2>&1 | tail -3
%cd /content/Code
!pip install -q -e . 2>&1 | tail -3

In [ ]:
# 5. Sanity-check cohort version distribution
%cd /content/Code
!python3 scripts/check_features_versions.py | tail -5

In [ ]:
# 6. Run the cumulative fit (Exp 1-4 in sequence, ~30-40 min)
%cd /content/Code
!python3 scripts/fit_aim2_phase1_batch.py --exp all 2>&1 | tee /content/phase1_batch.log | tail -200

In [ ]:
# 7. Sync results back to Drive
import shutil, os
SRC = '/content/Code/models/aim2_phase1_batch'
DST = f'{DRIVE_ROOT}/results/aim2_phase1_batch'
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
shutil.copy('/content/phase1_batch.log', f'{DRIVE_ROOT}/logs/phase1_batch_{int(__import__("time").time())}.log')
print(f'Results synced to: {DST}')
for sub in sorted(os.listdir(DST)):
    p = os.path.join(DST, sub)
    if os.path.isdir(p):
        files = sorted(os.listdir(p))
        print(f'  {sub}/ → {len(files)} files')

In [ ]:
# 8. Print compact summary
import json
summary = json.load(open(f'{SRC}/summary.json'))
print('Phase 1 cumulative ablation —', summary['cohort']['features_version'])
print(f"  cohort: {summary['cohort']['n_subjects_total']} subj, {summary['cohort']['n_epochs_total']:,} epochs (sleep-only)")
print()
print(f"{'Config':6s}  {'Features':>9s}  {'Test AUC':>9s}  {'Subject 95% CI':>22s}  {'AUC-PR':>8s}  {'F1':>7s}")
for name, r in summary['configs'].items():
    ci = f"[{r['test_auc_ci_low']:.4f}, {r['test_auc_ci_high']:.4f}]"
    print(f"{name:6s}  {r['n_features']:>9d}  {r['test_auc_roc']:>9.4f}  {ci:>22s}  {r['test_auc_pr']:>8.4f}  {r['test_f1_tuned']:>7.4f}")